# STAR Door-State Classifier - Google Colab Training

Fine-tunes **YOLOv8n-cls** on the **DoorDetect-Class-Dataset** to classify a rectified camera frame as `open`, `closed`, or `ajar`.
Outputs `door_state_yolov8n.onnx` ready to drop into `final_capstone/compliance-engine/star_compliance/models/`.

**Runtime:** Runtime -> Change runtime type -> Hardware accelerator: GPU (T4 is fine, ~15-20 min end to end).

**Sections:**
1. Verify GPU
2. Install dependencies
3. Download dataset from Google Drive
4. Normalize dataset layout (`semi-open` -> `ajar`)
5. Train YOLOv8n-cls
6. Validate on held-out test split
7. Export to ONNX (opset 12, simplified)
8. Download the ONNX file to your local machine

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install --quiet \
    'ultralytics>=8.3.0' \
    'onnx>=1.16' \
    'onnxruntime>=1.17' \
    'onnxsim>=0.4.35' \
    'opencv-python>=4.9' \
    'gdown>=5.0'

## 3. Paths and hyperparameters

Tweak `EPOCHS`, `IMG_SIZE`, `BATCH_SIZE` here if you want to experiment.

In [ ]:
import os

WORK_DIR = '/content/door_state_training'
DATASET_ROOT = f'{WORK_DIR}/dataset'
RUNS_DIR = f'{WORK_DIR}/runs'
OUTPUT_ONNX = f'{WORK_DIR}/door_state_yolov8n.onnx'

EPOCHS = 50
IMG_SIZE = 320
BATCH_SIZE = 64
DEVICE = 0

GDRIVE_URL = 'https://drive.google.com/drive/folders/1nI9rtgPbh25qh14vKXQvBpI4S1Szzukk'

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(DATASET_ROOT, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)
%cd {WORK_DIR}

## 4. Download the DoorDetect-Class-Dataset

Pulled from the author's Google Drive (~300 MB compressed, ~1 GB extracted).
If gdown hits a rate-limit, download the folder manually from
https://github.com/gasparramoa/DoorDetect-Class-Dataset#download and unzip into `dataset-raw/`.

In [ ]:
if not os.path.isdir(f'{WORK_DIR}/dataset-raw'):
    !gdown --folder '{GDRIVE_URL}' -O dataset-raw
else:
    print('dataset-raw already present, skipping download')

## 5. Normalize dataset layout

YOLOv8 classification expects:

```
dataset/
  train/{open,closed,ajar}/*.jpg
  val/{open,closed,ajar}/*.jpg
  test/{open,closed,ajar}/*.jpg
```

The dataset ships as `Cropped/{Train,Val,Test}/{open,closed,semi-open}`. Rename `semi-open` -> `ajar` to match what the STAR compliance engine expects.

In [ ]:
import shutil
from pathlib import Path

cropped_root = None
for p in Path(f'{WORK_DIR}/dataset-raw').rglob('Cropped'):
    if p.is_dir():
        cropped_root = p
        break

if cropped_root is None:
    raise RuntimeError('Cropped/ folder not found under dataset-raw/')
print(f'Found Cropped root at: {cropped_root}')

split_map = {'Train': 'train', 'Val': 'val', 'Test': 'test'}
class_map = {'closed': 'closed', 'open': 'open', 'semi-open': 'ajar'}

for split_src, split_dst in split_map.items():
    for cls_src, cls_dst in class_map.items():
        src_dir = cropped_root / split_src / cls_src
        dst_dir = Path(DATASET_ROOT) / split_dst / cls_dst
        dst_dir.mkdir(parents=True, exist_ok=True)
        if not src_dir.exists():
            print(f'  skip (missing): {src_dir}')
            continue
        for f in src_dir.iterdir():
            if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                shutil.copy2(f, dst_dir / f.name)

print('\nImages per split/class:')
for split in ['train', 'val', 'test']:
    for cls in ['open', 'closed', 'ajar']:
        count = sum(1 for _ in (Path(DATASET_ROOT) / split / cls).glob('*'))
        print(f'  {split:<5} / {cls:<7}  {count:4d} images')

## 6. Train YOLOv8n-cls

Metrics to watch in the per-epoch output:

| Metric | Target |
|---|---|
| top1 accuracy | >= 0.85 |
| top1 (open) | >= 0.90 |
| top1 (closed) | >= 0.90 |
| top1 (ajar) | >= 0.75 |

If top1 stalls below 0.80, rerun with `IMG_SIZE=416 EPOCHS=75 BATCH_SIZE=32` - ajar is the hardest class (only ~150 examples).

In [ ]:
!yolo classify train \
    model=yolov8n-cls.pt \
    data={DATASET_ROOT} \
    epochs={EPOCHS} \
    imgsz={IMG_SIZE} \
    batch={BATCH_SIZE} \
    device={DEVICE} \
    patience=15 \
    project={RUNS_DIR} \
    name=door_state_yolov8n \
    exist_ok=true \
    save_period=10

## 7. Validate on held-out test split

Acceptance thresholds for the capstone:
- Overall top-1 accuracy: >= 0.80
- Per-class top-1: open >= 0.85, closed >= 0.85, ajar >= 0.70

In [ ]:
BEST_PT = f'{RUNS_DIR}/door_state_yolov8n/weights/best.pt'
assert os.path.isfile(BEST_PT), f'Expected best.pt at {BEST_PT}'

!yolo classify val \
    model={BEST_PT} \
    data={DATASET_ROOT} \
    imgsz={IMG_SIZE} \
    split=test \
    device={DEVICE} \
    project={RUNS_DIR} \
    name=door_state_eval \
    exist_ok=true

## 8. Export to ONNX

Opset 12 with simplification; matches what `door_state_classifier.py` expects at runtime.

In [ ]:
!yolo export \
    model={BEST_PT} \
    format=onnx \
    imgsz={IMG_SIZE} \
    opset=12 \
    simplify=true

BEST_ONNX = f'{RUNS_DIR}/door_state_yolov8n/weights/best.onnx'
assert os.path.isfile(BEST_ONNX), f'ONNX export failed; expected {BEST_ONNX}'

shutil.copy2(BEST_ONNX, OUTPUT_ONNX)
print(f'Copied: {OUTPUT_ONNX}')

## 9. md5 + size report

Record these in `final_capstone/compliance-engine/star_compliance/models/README.md` alongside the trained weights.

In [ ]:
import hashlib

def md5_of(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

size_bytes = os.path.getsize(OUTPUT_ONNX)
digest = md5_of(OUTPUT_ONNX)
print(f'ONNX:  {OUTPUT_ONNX}')
print(f'Size:  {size_bytes} bytes')
print(f'md5:   {digest}')

## 10. Download ONNX to your local machine

Drop the downloaded file into
`final_capstone/compliance-engine/star_compliance/models/door_state_yolov8n.onnx`
in the STAR repo, update `models/README.md` with the md5 + dataset commit, run
`pytest tests/` in the compliance-engine package, then commit and push.

In [ ]:
from google.colab import files
files.download(OUTPUT_ONNX)